# Day 1 / Session 3 -- Model Evaluation Demos

This notebook contains the **complete demo walkthrough** for Session 3.
You will see seven demos that build a full evaluation toolkit:

| # | Demo | What it builds |
|---|------|---------------|
| 1 | Scoring Rubric | Structured 1-5 rubric with consulting-grade criteria |
| 2 | LLM Judge | Automated evaluation that returns JSON scores + reasoning |
| 3 | Benchmarking | Side-by-side quality/latency comparison across configs |
| 4 | Cost Analysis | Per-token cost estimation and performance summaries |
| 5 | A/B Testing | Framework to run controlled experiments on two configs |
| 6 | Sklearn Metrics | Precision, recall, F1, and confusion matrix for LLM classification |
| 7 | G-Eval | Chain-of-thought LLM judge (research-backed scoring) |

**Your role:** you are the engineer building and operating this evaluation system.
Consultants and partners are the end-users whose deliverable quality you are measuring.

## Session Goal

By the end of this session you will have built a **complete evaluation toolkit** from scratch:

- **Scoring rubrics** that define what "good" looks like for consulting AI outputs
- **Automated LLM-as-Judge** that scores responses at scale without human reviewers
- **Benchmarking harness** that compares model configurations side-by-side
- **Cost analysis utilities** that forecast production spend per token
- **A/B testing framework** for controlled experiments between two configs
- **Classification metrics** (precision, recall, F1, confusion matrix) using scikit-learn
- **G-Eval** -- a research-backed chain-of-thought scoring method that correlates highly with human judgment

These tools ensure that AI systems deployed for consultants meet McKinsey quality standards.
Every evaluation approach you build here maps directly to a production deployment concern:
quality, cost, consistency, and auditability.

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn langchain langchain-openai langchain-core python-dotenv

## Environment Setup

Run this cell once to load credentials and import libraries.

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

import openai
import json
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

---
## Demo 1: Setting Up Evaluation Metrics and Scoring Rubrics

Before comparing models, we need a consistent framework for measuring quality.
This demo defines five consulting-grade evaluation criteria and builds a
structured scoring rubric with descriptions for every level (1-5).

As the engineer, you will wire this rubric into every downstream evaluation
so that consultants and partners get consistent, transparent feedback.

In [ ]:
client = openai.OpenAI()

# Define evaluation criteria for consulting deliverables
evaluation_criteria = {
    "strategic_relevance": "How relevant is the analysis to the client's strategic question? (1-5)",
    "analytical_rigor": "How data-driven and methodologically sound is the analysis? (1-5)",
    "mece_structure": "How well-structured and MECE is the response? (1-5)",
    "actionability": "How actionable are the recommendations for the client? (1-5)",
    "executive_readiness": "Is this ready for a C-suite presentation? (1-5)"
}

def create_scoring_rubric():
    rubric = {}
    for criterion, description in evaluation_criteria.items():
        rubric[criterion] = {
            "description": description,
            "1": "Poor - Fails to meet basic expectations for consulting deliverables",
            "2": "Below Average - Partially addresses the criterion; not client-ready",
            "3": "Average - Adequately meets the criterion but lacks McKinsey-level polish",
            "4": "Good - Exceeds expectations; approaching partner-quality output",
            "5": "Excellent - Exceptional quality; ready for CEO/board presentation"
        }
    return rubric

rubric = create_scoring_rubric()
for criterion, details in rubric.items():
    print(f"\n{criterion.upper()}: {details['description']}")
    for score, desc in details.items():
        if score != "description":
            print(f"  {score}: {desc}")

---
## Demo 2: Automated Evaluation with LLM-as-a-Judge

Manual scoring does not scale. This demo builds an `llm_judge` function that
sends a response to GPT along with the rubric criteria and gets back structured
JSON scores with short reasoning for each criterion.

The system prompt frames the judge as a consulting engagement manager --
the same persona that partners use when reviewing deliverables.

In [ ]:
# Demo 2a: Define the LLM judge function

def llm_judge(question, response_text, criteria):
    """Score a response using LLM-as-a-Judge. Returns {criterion: {score, reasoning}}."""
    criteria_list = "\n".join(f"- {k}: {v}" for k, v in criteria.items())
    judge_prompt = f"""You are a McKinsey engagement manager evaluating AI-generated consulting analysis.
Rate the following response on a scale of 1-5 for each criterion.

Client question: {question}

Response to evaluate:
{response_text}

Criteria:
{criteria_list}

Return ONLY a JSON object. Each key is a criterion name, each value is an object with "score" (integer 1-5) and "reasoning" (one short sentence).
Keep reasoning under 15 words per criterion to stay concise."""

    client = openai.OpenAI()
    evaluation = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": judge_prompt}],
        response_format={"type": "json_object"},
        max_tokens=600
    )
    return json.loads(evaluation.choices[0].message.content)

print("llm_judge() defined -- ready to score responses.")

**What's happening here:** The `llm_judge` function sends the response being evaluated, along with the scoring criteria, to an LLM and asks it to play the role of a McKinsey engagement manager. The JSON response format ensures we get structured, machine-readable scores.

In [ ]:
# Demo 2b: Test the judge with a consulting scenario

test_question = "How should a mid-size retailer approach digital transformation to improve margins?"
test_response = """To improve margins through digital transformation, a mid-size retailer should focus on three key levers:

1. **Customer Analytics & Personalization**: Implement a CDP (Customer Data Platform) to unify purchase history, browsing behavior, and demographics. Use predictive analytics to personalize promotions, reducing blanket discounting by 15-20% while maintaining conversion rates.

2. **Supply Chain Digitization**: Deploy demand forecasting models to reduce inventory carrying costs by 10-15%. Integrate real-time POS data with procurement to minimize stockouts and overstock situations.

3. **Omnichannel Integration**: Unify online and offline channels through BOPIS (Buy Online, Pick Up In Store) and ship-from-store capabilities. This typically improves inventory turns by 20-30% and increases customer lifetime value.

Implementation should follow a phased 18-month roadmap, starting with quick wins in analytics (months 1-6), then supply chain (months 6-12), and finally full omnichannel (months 12-18). Expected margin improvement: 200-400 basis points."""

result = llm_judge(test_question, test_response, evaluation_criteria)
print(json.dumps(result, indent=2))

**Observe:** Check which criteria scored highest/lowest. Does the scoring align with your own assessment? LLM judges correlate well with human evaluators but can have blind spots -- that's why G-Eval (Demo 7) adds chain-of-thought reasoning.

### Try This
Modify the `test_response` to be deliberately vague and generic (remove all numbers and specific recommendations). Re-run the judge. How do the scores change? Which criteria drop the most?

---
## Demo 3: Benchmarking Response Quality Across Models

With the judge in place, we can now benchmark multiple model configurations
against the same set of prompts. This demo collects response length, latency,
and token usage across three temperature settings.

As the platform engineer, you will use these benchmarks to decide which config
to deploy behind the consulting tool that partners interact with.

In [ ]:
# Demo 3a: Run benchmarks across configurations

test_prompts = [
    "What are the key value creation levers in a post-merger integration?",
    "How should a CPG company respond to private label competition?",
    "What framework would you use to evaluate market entry into Southeast Asia?"
]

configurations = [
    {"model": "gpt-4o-mini", "temperature": 0.0, "label": "GPT-4o-mini (T=0)"},
    {"model": "gpt-4o-mini", "temperature": 0.7, "label": "GPT-4o-mini (T=0.7)"},
    {"model": "gpt-4o-mini", "temperature": 1.0, "label": "GPT-4o-mini (T=1.0)"},
]

results = []
for config in configurations:
    for prompt in test_prompts:
        start_time = time.time()
        response = client.chat.completions.create(
            model=config["model"],
            temperature=config["temperature"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        elapsed = time.time() - start_time
        content = response.choices[0].message.content
        results.append({
            "config": config["label"],
            "prompt": prompt[:50] + "...",
            "response_length": len(content),
            "latency_ms": round(elapsed * 1000),
            "tokens_used": response.usage.total_tokens
        })

print(f"Benchmarks collected: {len(results)} runs")

In [ ]:
# Demo 3b: Display benchmark results as a DataFrame

df = pd.DataFrame(results)
print(df.to_string(index=False))

**Gotcha:** Response length varies but token usage is nearly identical across temperature settings -- that's because `max_tokens` caps the output. Latency differences are mostly network variation, not model behavior.

### Quick Recap
- **Q:** Why is JSON output format important for the LLM judge? What would go wrong with free-text scoring?
- **Q:** If two model configs produce similar benchmark scores, what other factors should influence your choice?

---
## Demo 4: Latency and Cost Analysis

Understanding cost and latency is critical when you are building a system that
consultants will use dozens of times per engagement. This demo provides utility
functions for per-token cost estimation and aggregate performance summaries.

In [ ]:
# Demo 4a: Define pricing table and cost estimation function

PRICING = {
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
}

def estimate_cost(model, input_tokens, output_tokens):
    if model in PRICING:
        input_cost = input_tokens * PRICING[model]["input"]
        output_cost = output_tokens * PRICING[model]["output"]
        return round(input_cost + output_cost, 6)
    return None

# Quick sanity check
for model_name, prices in PRICING.items():
    cost = estimate_cost(model_name, 500, 300)
    print(f"{model_name}: Estimated cost for 500 input + 300 output tokens = ${cost}")

In [ ]:
# Demo 4b: Aggregate performance analysis

def analyze_performance(results_df):
    summary = results_df.groupby("config").agg({
        "latency_ms": ["mean", "min", "max"],
        "response_length": "mean",
        "tokens_used": "sum"
    }).round(2)
    print("Performance Summary:")
    print(summary)
    return summary

if len(results) > 0:
    analyze_performance(df)

**Key Insight:** gpt-4o is ~17x more expensive than gpt-4o-mini. For evaluation pipelines that run hundreds of scoring calls, model choice has major cost implications. Use mini for drafts, full model for final quality gates.

---
## Demo 5: A/B Testing Framework

A/B testing lets you rigorously compare two model configurations by running
them side-by-side on the same prompts and evaluating both with the LLM judge.

In production you would run this as a background job and surface results in
a dashboard that engineering and engagement leadership can review together.

In [ ]:
# Demo 5a: Define the A/B Testing Framework class

import random

class ABTestFramework:
    def __init__(self, config_a, config_b):
        self.config_a = config_a
        self.config_b = config_b
        self.results_a = []
        self.results_b = []
        self.client = openai.OpenAI()

    def run_test(self, prompts, judge_criteria):
        for prompt in prompts:
            # Get responses from both configurations
            resp_a = self.client.chat.completions.create(
                model=self.config_a["model"],
                temperature=self.config_a.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_a.get("system", "You are a McKinsey senior consultant. Provide structured, MECE, data-driven strategic analysis.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            resp_b = self.client.chat.completions.create(
                model=self.config_b["model"],
                temperature=self.config_b.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_b.get("system", "You are a McKinsey senior consultant. Provide structured, MECE, data-driven strategic analysis.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )

            score_a = llm_judge(prompt, resp_a.choices[0].message.content, judge_criteria)
            score_b = llm_judge(prompt, resp_b.choices[0].message.content, judge_criteria)
            self.results_a.append(score_a)
            self.results_b.append(score_b)
            print(f"Prompt: {prompt[:50]}... | Config A vs B evaluated")

    def get_summary(self):
        print(f"\nA/B Test Results:")
        print(f"Config A: {self.config_a.get('label', 'A')}")
        print(f"Config B: {self.config_b.get('label', 'B')}")
        print(f"Tests run: {len(self.results_a)}")

print("ABTestFramework class defined.")

In [ ]:
# Demo 5b: Initialize and show usage

ab_test = ABTestFramework(
    config_a={"model": "gpt-4o-mini", "temperature": 0.0, "label": "Deterministic"},
    config_b={"model": "gpt-4o-mini", "temperature": 0.7, "label": "Creative"}
)
print("A/B Testing Framework initialized!")
print("Run ab_test.run_test(prompts, criteria) to execute tests")

### Quick Recap
- **Q:** When would you use sklearn metrics (Demo 6) vs LLM-as-Judge (Demo 2)? They solve different problems -- what's the distinction?
- **Q:** An A/B test shows Config A scores 4.2 and Config B scores 4.0. Is that difference meaningful? What else would you need to know?

---
## Demo 6: Scikit-learn Metrics -- Precision, Recall, F1 for LLM Classification

When LLMs perform classification tasks (sentiment, intent, category), you can
evaluate their accuracy using standard scikit-learn metrics. This gives you
quantitative, reproducible scores beyond subjective quality rubrics.

| Metric | What it measures |
|--------|------------------|
| **Precision** | Of all items predicted as class X, how many were actually X? |
| **Recall** | Of all actual class X items, how many did we correctly predict? |
| **F1 Score** | Harmonic mean of precision and recall -- balances both |

In [ ]:
# Demo 6a: Define the labeled test dataset -- client engagement feedback (ground truth)

client = openai.OpenAI()

test_data = [
    {"text": "The engagement exceeded expectations -- insights transformed our strategy.", "true_label": "POSITIVE"},
    {"text": "The team's frameworks were razor-sharp and the final readout was outstanding.", "true_label": "POSITIVE"},
    {"text": "McKinsey brought world-class analytical rigor that accelerated our transformation.", "true_label": "POSITIVE"},
    {"text": "The partner's leadership and team's dedication delivered exceptional value.", "true_label": "POSITIVE"},
    {"text": "Deliverables were late and analysis didn't address core issues.", "true_label": "NEGATIVE"},
    {"text": "The recommendations were impractical -- clearly the team didn't understand our industry.", "true_label": "NEGATIVE"},
    {"text": "We spent millions and the output was a deck we could have built internally.", "true_label": "NEGATIVE"},
    {"text": "Communication was poor and the project went over budget with no clear ROI.", "true_label": "NEGATIVE"},
    {"text": "The team was professional but recommendations were generic.", "true_label": "NEUTRAL"},
    {"text": "Solid work overall, though nothing we hadn't considered before.", "true_label": "NEUTRAL"},
    {"text": "The analysis was competent but didn't push our thinking in new directions.", "true_label": "NEUTRAL"},
    {"text": "Reasonable engagement -- met the brief but didn't go above and beyond.", "true_label": "NEUTRAL"},
]

print(f"Test dataset: {len(test_data)} labeled feedback samples")
print(f"Classes: {set(d['true_label'] for d in test_data)}")

In [ ]:
# Demo 6b: Classify each feedback sample with the LLM

true_labels = []
predicted_labels = []

for item in test_data:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a client feedback classifier. Classify the sentiment of client engagement feedback. Respond with EXACTLY one word: POSITIVE, NEGATIVE, or NEUTRAL."},
            {"role": "user", "content": f"Classify this client feedback: '{item['text']}'"}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "5")),
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
    )
    prediction = response.choices[0].message.content.strip().upper()
    true_labels.append(item["true_label"])
    predicted_labels.append(prediction)

print("Classification complete.")
print(f"Predictions: {predicted_labels}")

**Observe:** Look at the confusion matrix. NEUTRAL is typically the hardest category -- the model may conflate 'neutral' with 'slightly positive' or 'slightly negative'. This is a common challenge in sentiment classification.

In [ ]:
# Demo 6c: Compute scikit-learn metrics and classification report

labels = ["POSITIVE", "NEGATIVE", "NEUTRAL"]

print("CLASSIFICATION REPORT -- Client Feedback")
print("=" * 60)
print(classification_report(true_labels, predicted_labels, labels=labels))

# Individual weighted metrics
precision = precision_score(true_labels, predicted_labels, labels=labels, average="weighted")
recall = recall_score(true_labels, predicted_labels, labels=labels, average="weighted")
f1 = f1_score(true_labels, predicted_labels, labels=labels, average="weighted")

print(f"\nWeighted Precision: {precision:.4f}")
print(f"Weighted Recall   : {recall:.4f}")
print(f"Weighted F1 Score : {f1:.4f}")

In [ ]:
# Demo 6d: Confusion matrix visualization

cm = confusion_matrix(true_labels, predicted_labels, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix -- Client Feedback Classification")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()

print("\nObservation: The confusion matrix shows where the LLM agrees/disagrees with ground truth.")
print("Diagonal = correct predictions. Off-diagonal = misclassifications.")

**Gotcha:** These metrics assume your ground-truth labels are correct. In practice, even human annotators disagree on edge cases. Always validate your test dataset with multiple reviewers.

### Try This
Add 2-3 more tricky/ambiguous feedback statements to `test_data`. How does accuracy change? Which category has the most misclassifications?

---
## Demo 7: G-Eval -- LLM-as-Judge with Chain-of-Thought Scoring

[G-Eval](https://arxiv.org/abs/2303.16634) is a research-backed evaluation
method that uses an LLM with **chain-of-thought reasoning** to score text
quality. It achieves higher correlation with human judgments than traditional
metrics.

The approach:
1. Define evaluation criteria (coherence, fluency, relevance)
2. Ask the LLM to think step-by-step before assigning a score
3. Extract the numeric score from the structured response

This demo implements G-Eval from scratch using the OpenAI API so you can
see exactly how it works and integrate it into your evaluation pipeline.

In [ ]:
# Demo 7a: Define G-Eval function and evaluation criteria

def g_eval(input_text, output_text, criterion_name, criterion_description):
    """Score text using G-Eval: chain-of-thought LLM evaluation.

    Returns dict with score (0.0-1.0), reasoning, and raw response.
    """
    prompt = f"""You are an expert evaluator assessing AI-generated consulting analysis.

Evaluation Criterion: {criterion_name}
Description: {criterion_description}

Input/Question: {input_text}

Output to Evaluate:
{output_text}

Instructions:
1. First, think step-by-step about how well the output meets the criterion.
2. Then assign a score from 1 to 5.

Respond in JSON with keys: "reasoning" (your step-by-step analysis, 2-3 sentences) and "score" (integer 1-5)."""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=300
    )
    result = json.loads(response.choices[0].message.content)
    # Normalize score to 0-1 range
    result["normalized_score"] = result["score"] / 5.0
    return result


# Define consulting-quality evaluation criteria
g_eval_criteria = {
    "Coherence": "The response should be logically organized with a clear storyline, following a structured consulting framework (e.g., situation-complication-resolution).",
    "Fluency": "The response should be grammatically correct, well-written, and ready for a client-facing consulting deliverable.",
    "Relevance": "The response should directly address the client's strategic question and provide actionable consulting insights.",
}

print("g_eval() defined with 3 criteria: Coherence, Fluency, Relevance")

**Key Insight:** G-Eval uses chain-of-thought reasoning BEFORE assigning a score. This mirrors how a McKinsey partner thinks through a review -- first analyze, then score -- and research shows it produces more calibrated ratings.

In [ ]:
# Demo 7b -- Part A: G-Eval on a consulting response

print("PART A: G-Eval -- Evaluating consulting output quality")
print("=" * 60)

client = openai.OpenAI()
test_question = "Explain the benefits of a hub-and-spoke operating model for a global manufacturer."
test_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": test_question}],
    max_tokens=300
).choices[0].message.content

print(f"Question: {test_question}")
print(f"Response: {test_response[:200]}...\n")

for criterion_name, criterion_desc in g_eval_criteria.items():
    result = g_eval(test_question, test_response, criterion_name, criterion_desc)
    print(f"  {criterion_name}: {result['score']}/5 ({result['normalized_score']:.2f})")
    print(f"    Reasoning: {result['reasoning'][:100]}...")

In [ ]:
# Demo 7c -- Part B: Answer Relevancy -- Post-Merger Integration

print("PART B: Answer Relevancy -- Post-Merger Integration")
print("=" * 60)

pmi_question = "What are best practices for post-merger integration in financial services?"
pmi_response = """Post-merger integration in financial services should prioritize four areas:
(1) Day-1 readiness for regulatory compliance and customer continuity,
(2) technology platform consolidation with a focus on core banking systems,
(3) cultural integration through joint leadership workshops and unified incentive structures,
(4) synergy capture tracking with a dedicated PMI office reporting to the CEO.
Quick wins in cost synergies (branch rationalization, vendor consolidation) should be pursued in the first 100 days."""

relevancy_result = g_eval(pmi_question, pmi_response, "Relevance",
    "The response should directly and comprehensively address the specific question asked.")
print(f"Relevancy Score: {relevancy_result['score']}/5 ({relevancy_result['normalized_score']:.2f})")
print(f"Passed threshold (0.7): {relevancy_result['normalized_score'] >= 0.7}")
print(f"Reasoning: {relevancy_result['reasoning']}")

In [ ]:
# Demo 7d -- Part C: Comparing responses across temperature settings

print("PART C: Comparing consulting responses across temperature settings")
print("=" * 60)

question = "What are the key considerations for a private equity portfolio company's digital transformation?"
comparison_results = []

for temp in [0.0, 0.5, 1.0]:
    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": question}],
        temperature=temp,
        max_tokens=300
    ).choices[0].message.content

    coherence = g_eval(question, resp, "Coherence", g_eval_criteria["Coherence"])
    relevance = g_eval(question, resp, "Relevance", g_eval_criteria["Relevance"])

    comparison_results.append({
        "temperature": temp,
        "coherence": coherence["normalized_score"],
        "relevance": relevance["normalized_score"],
        "response_length": len(resp)
    })
    print(f"  T={temp}: coherence={coherence['normalized_score']:.2f}, relevance={relevance['normalized_score']:.2f}")

results_df = pd.DataFrame(comparison_results)
print(f"\n{results_df.to_string(index=False)}")

**Observe:** Do higher temperatures produce lower coherence scores? Not always -- temperature affects style variety more than logical structure. The real impact shows up when you run the same evaluation multiple times.

### Quick Recap
- **Q:** How does G-Eval differ from the basic LLM judge in Demo 2?
- **Q:** You're evaluating an AI system that generates M&A due diligence reports. Which evaluation approach(es) from this session would you combine, and why?

---
## Demo Recap

You now have a complete evaluation toolkit:

1. **Scoring Rubric** -- structured 1-5 criteria that every evaluation references
2. **LLM Judge** -- automated JSON scoring that scales to thousands of evaluations
3. **Benchmarking** -- side-by-side comparison of model configs on quality + latency
4. **Cost Analysis** -- per-token cost estimation so you can forecast production spend
5. **A/B Testing** -- controlled experiments for statistically confident model selection
6. **Sklearn Metrics** -- precision, recall, F1 for hard classification tasks
7. **G-Eval** -- chain-of-thought LLM judge that correlates with human judgments

Next up: open `D1S3_exercises.ipynb` to build your own custom evaluation rubric.